# Exercise: Structures of filters

Until now, LTI-systems have been described by coefficients of vectors `b` and `a`, which matches the usual form of description using the system or Z transfer function

$$
H(z) = \frac{b_0 + b_1 z^{-1} + b_2 z^{-2} + \cdots}{1+ a_1 z^{-1} + a_2 z^{-2} + \cdots} \,\,\,.
$$

Another (equivalent) description may be done with poles and zeros,

$$
H(z) = b_0 \frac{(z-z_{0,1})(z-z_{0,2})(z-z_{0,3})\cdots}{(z-z_{\infty,1})(z-z_{\infty,2})(z-z_{\infty,3})\cdots} \qquad.
$$

LTI-systems can be implemented using *canonical* structures. Commonly, a canonical structure is a form of realization with a minimal number of memory elements. You may look at chapter~4.1 for the descriptions of the particular implementation. The *third canonical* structure, which is examined in this exercise, allows us to transform the system function to a pole-zero form directly.

The distinctive poles resp. zeros are grouped into sections of maximal second order (SOS = second order section) and cascaded in a network. The main benefits of this structure are the flexibility of the sequence as well as good behaviour when using quantized parameters.

\footnote{For the subject of parameter quantization, see section~4.4 resp.~exercise \ref{dsv4a04e}.}

\matlab-function `tf2zp()` converts the polynomial form to the pole-zero form. Using the pole-zero form, MATLAB’s function `zp2sos()` can calculate the second order sections of the third canonical form.

---

## a)

Write a new MATLAB function `[B A] = lcascade(b,a)` that builds sections of second order out of filter vectors `b` and `a`. The coefficients of the second order sections may be calculated using `tf2zp()` and `zp2sos()` and should be stored in row-vectors of matrices `B` and `A`.

In [ ]:
# BEGIN SOLUTION
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ellip, freqz
from scipy.signal import tf2zpk, zpk2sos


def lcascade(b, a):
    z, p, k = tf2zpk(b, a)
    sos = zpk2sos(z, p, k)
    B = sos[:, :3]
    A = sos[:, 3:6]
    
    return B, A 
# END SOLUTION

## b)

Calculate (using `lcascade()`) the filter coefficients of the second order sections of a 7th order Cauer lowpass with the following specifications:

- $3\,\text{dB}$-cut off frequency: $0.4\,\pi$
- pass band attenuation: $R_P = 0.1\,\text{dB}$
- stop band attenuation: $R_S = 40\,\text{dB}$

The tolerance parameters $R_S$ resp.~$R_P$ are given as follows:

$$
R_P = -20\log_{10}(1 - \delta_D), \quad
R_S = -20\log_{10}(\delta_S),
$$

whereby $\delta_D$ resp. $\delta_S$ are the tolerance parameters as introduced in chapter~4.

MATLAB’s function `ellip(7, 0.1, 40, 0.4)` yields the filter coefficients:

```matlab
[b, a] = ellip(7, 0.1, 40, 0.4);
[B, A] = lcascade(b, a);
```

Calculate the transfer function of the complete system corresponding to:

$$
H(z) = \prod_{i=1}^{p} H_i(z).
$$

In [ ]:
# BEGIN SOLUTION
b, a = ellip(7, 0.1, 40, 0.4)  # Design filter

B, A = lcascade(b, a)  # 3rd-order canonical form

m, n = B.shape
Hi = np.zeros((512, m), dtype=complex)

# Use the same frequency vector for all partial responses
# Define frequency axis
W = np.linspace(0, np.pi, 512)

# Whole system response on fixed axis
_, Hg = freqz(b, a, 512)

# Subsystem responses also on fixed axis
Hi = np.zeros((512, m), dtype=complex)
for k in range(m):
    _, H_temp = freqz(B[k, :], A[k, :], worN=W)
    Hi[:, k] = H_temp

H3k = np.prod(Hi, axis=1)

plt.figure()
plt.plot(W / np.pi, 20 * np.log10(np.abs(H3k)), '--b', label='Product of Sub-systems')
plt.plot(W / np.pi, 20 * np.log10(np.abs(Hg)), '-.b', label='Whole System')
plt.grid()
plt.xlabel('Omega/pi (Nyquist = 1)')
plt.ylabel('|H(exp(j*Omega))| in dB')
plt.title('-- Whole System, -.Product of Sub-systems')
plt.ylim([-100, None])
plt.legend()
plt.show()
#END SOLUTION